# Procesamiento de Alto Volumen de Datos
## Medición de Calidad del Agua
La calidad del agua representa la calidad de vida de los seres humanos. El agua es junto con el aire, las necesidades más importantes para la vida. La definición popular de calidad del agua son sus características físicas, químicas y biológicas del agua [1]. El presente estudio tiene como objetivo el análisis de la calidad del agua en la India, como metodología de trabajo sobre estudios de comportamiento de contaminación y sus modelos en ML para la predicción.

In [0]:
import pyspark.sql.functions as F

In [0]:
df = spark.sql("SELECT * FROM workspace.waterquality.waterquality")

In [0]:
df00 = spark.table("workspace.waterquality.waterquality")

In [0]:
df00.show(5)

In [0]:
df00.printSchema()

Se presentan los parámetros y sus definiciones de la calidad del agua en las regiones de la India. La definición de las fuentes, impactos, efectos y métodos de medida de la base de datos: no son contempladas en el presente estudio. Las definiciones de los parámetros dados por la base de datos son extraidos de **[2]**.

- STATION CODE: Código de estación
- LOCATIONS: Ubicación de estación (Ubicación del Rio)
- STATE: Estado/Lugar de la India
- TEMP: Temperatura promedio del agua (°C)
- DO: Oxigeno Disuelto (mg/L). Concentraciones altas de oxigeno disuelto representa mejor calidad del agua.
- pH: Se define como el logaritmo negativo de la concentración de hidrógeno. Número adimensional que indica la acidez o base de una solución **[3]**.
- CONDUCTIVITY: Mide la habilidad de una solución conducir corriente electrica **[4]**. El agua pura no es buen conductor de electricidad **[5]**.
- BOD: Las bacterias y otros microorganismos utilizan sustancias orgánicas como alimento (Demanda Bioquímica de Oxigeno). A medida que metabolizan la materia orgánica, consumen oxígeno **[4]**. Mayor cantidad de material orgánico en el agua, mayor valor de BOD.
- 'NITRATE_N_NITRITE_N': Nitrito y Nitrato Nitrogeno. Altas concentraciones de N en la superficie del agua puede estimular el rápido crecimiento de algas las cuales degradan la calidad del agua (mg/L). **[4]**.
- 'FECAL_COLIFORM': Promedio de bacterias coliforms (excresiones) **[6]** .
-  'TOTAL_COLIFORM: Cantidad total de coliformes. Se retira del dataset ya que no aporta información promedio sobre los datos.

### Limpieza de datos

In [0]:
df01 = df00

In [0]:
df00.columns

In [0]:
cols_to_fix = ['TEMP','DO','pH','CONDUCTIVITY','BOD','NITRATE_N_NITRITE_N','FECAL_COLIFORM','TOTAL_COLIFORM']

for col_name in cols_to_fix:
    df01 = df01.withColumn(col_name, df01[col_name].cast("float"))

In [0]:
# #Va a fallar por los valores "NA"
# df01.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df01.columns]).show()

In [0]:
df01 = df00

In [0]:
df01 = df00
cols_to_fix_0 = ['TEMP','DO','pH','CONDUCTIVITY','BOD','NITRATE_N_NITRITE_N','FECAL_COLIFORM','TOTAL_COLIFORM']

for col_name in cols_to_fix_0:
    df01 = df01.withColumn(col_name, F.expr(f"try_cast(`{col_name}` as float)"))

In [0]:
df01.printSchema()

In [0]:
df01.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df01.columns]).show()

In [0]:
df01 = df01.drop("TOTAL_COLIFORM")
df01.columns

#### Limpieza de Nulls en SQL

In [0]:
df01.createOrReplaceTempView("df01_tempview")

In [0]:
df01 = spark.sql('''SELECT *
                 FROM df01_tempview
                 WHERE TEMP IS NOT NULL
                 AND DO IS NOT NULL
                 AND pH IS NOT NULL
                 AND CONDUCTIVITY IS NOT NULL
                 AND BOD IS NOT NULL
                 AND FECAL_COLIFORM IS NOT NULL
                 AND NITRATE_N_NITRITE_N IS NOT NULL''')

In [0]:
df01.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df01.columns]).show()

%md
# **Ingeniería de Características**

**Water Quality Index:** El Índice de la Calidad del Agua se calcula agregando  linealmente el índice de calidad con el peso.

WQI = sum(**qr**n * **W**n)

**qr**n: Rango de Calidad para el n parámetro de calidad de agua.

**W**n: Unidad de peso para el n parámetro.

A continuación se calcula un método estándar **[2]** para calcular el rango de calidad **qr** para cada parámetro.

* Rango de calidad para **pH**. Se crea una nueva columna con los rangos **pH** de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`pH = [7.0 ; 8.5]`**
    + 80:  *agua moderada [parte baja o parte alta]* **`pH = [6.8 ; 6.9) o (8.5 ; 8.6)`**
    + 60:  *agua dura [parte baja o parte alta]* **`pH = [6.7 ; 6.8) o [8.6 ; 8.8)`**
    + 40:  *agua muy dura [parte baja o parte alta]* **`pH = [6.5 ; 6.7) o [8.8 ; 9.0)`**
    + 0:   rango agua residuales

* Rango de calidad para **Oxigeno Disuelto**. Se crea una nueva columna con los rangos DO de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`DO>=6.0`**
    + 80:  *agua moderada* **`DO = [5.0, 6.0)`**
    + 60:  *agua dura* **`DO = [4.0, 5.0)`**
    + 40:  *agua muy dura* **`DO = [3.0, 4.0)`**
    + 0:   rango agua residuales

* Rango de calidad para **Conductividad**. Se crea una nueva columna con los rangos CONDUCTIVITY de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`CONDUCTIVITY = [0.0,75.0)`**
    + 80:  *agua moderada* **`CONDUCTIVITY = [75.0, 150.0)`**
    + 60:  *agua dura* **`CONDUCTIVITY = [150.0, 225.0)`**
    + 40:  *agua muy dura* **`CONDUCTIVITY = [225.0, 300.0)`**
    + 0:   rango agua residuales

* Rango de calidad para **Demanda Bioquímica de Oxigeno**. Se crea una nueva columna con los rangos **BOD** de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`BOD = [0.0,3.0)`**
    + 80:  *agua moderada* **`BOD = [3.0, 6.0)`**
    + 60:  *agua dura* **`BOD = [6.0, 80.0)`**
    + 40:  *agua muy dura* **`BOD = [80.0, 125.0)`**
    + 0:   rango agua residuales

* Rango de calidad para **Nitratos**. Se crea una nueva columna con los rangos **NITRATE_N_NITRITE_N** de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`NITRATE_N_NITRITE_N = [0.0,20.0)`**
    + 80:  *agua moderada* **`NITRATE_N_NITRITE_N =  [20.0, 50.0)`**
    + 60:  *agua dura* **`NITRATE_N_NITRITE_N = [50.0, 100.0)`**
    + 40:  *agua muy dura* **`NITRATE_N_NITRITE_N = [100.0, 200.0)`**
    + 0:   rango agua residuales
      
* Rango de calidad para **Coliforme Fecal**. Se crea una nueva columna con los rangos FECAL_COLIFORM de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`FECAL_COLIFORM = [0.0,5.0)`**
    + 80:  *agua moderada* **`FECAL_COLIFORM = [5.0, 50.0)`**
    + 60:  *agua dura* **`FECAL_COLIFORM = [50.0, 500.0)`**
    + 40:  *agua muy dura* **`FECAL_COLIFORM = [500.0, 10000.0)`**
    + 0:   rango agua residuales


In [0]:
df01.show(5)

In [0]:
#Función para definir el rango de calidad de agua según el pH
# Se crea la columna para los rangos del parámetro pH
df02 = (
    df01
    .withColumn(
        "qrPH",
        F.when(
            (df01.pH >= 7.0) & (df01.pH <= 8.5),  # Agua dulce
            100
        )
        .when(
            (
                (df01.pH >= 6.8) & (df01.pH < 6.9)   # Agua moderada (parte baja)
            ) | (
                (df01.pH > 8.5) & (df01.pH < 8.6)   # Agua moderada (parte alta)
            ),
            80
        )
        .when(
            (
                (df01.pH >= 6.7) & (df01.pH < 6.8)  # Agua dura (parte baja)
            ) | (
                (df01.pH >= 8.6) & (df01.pH < 8.8)  # Agua dura (parte alta)
            ),
            60
        )
        .when(
            (
                (df01.pH >= 6.5) & (df01.pH < 6.7)  # Agua muy dura (parte baja)
            ) | (
                (df01.pH >= 8.8) & (df01.pH < 9.0)  # Agua muy dura (parte alta)
            ),
            40
        )
        .otherwise(0)  # Rango residual
    )
)

In [0]:
#Función para definir el rango de calidad de agua según el DO
# Se crea la columna para los rangos del parámetro DO
"""
* Rango de calidad para **Oxigeno Disuelto**. Se crea una nueva columna con los rangos DO de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`DO>=6.0`**
    + 80:  *agua moderada* **`DO = [5.0, 6.0)`**
    + 60:  *agua dura* **`DO = [4.0, 5.0)`**
    + 40:  *agua muy dura* **`DO = [3.0, 4.0)`**
    + 0:   rango agua residuales
"""
df02 = (
    df02
    .withColumn(
        "qrDO",
        F.when(
            (df01.DO >= 6),  # Agua dulce
            100
        )
        .when(
            (
                (df01.DO >= 5) & (df01.DO < 6)   # Agua moderada
            ),
            80
        )
        .when(
            (
                (df01.DO >= 4) & (df01.DO < 5) # Agua dura
            ),
            60
        )
        .when(
            (
                (df01.DO >= 3) & (df01.DO < 4)  # Agua muy dura
            ),
            40
        )
        .otherwise(0)  # Rango residual
    )
)

In [0]:
#Función para definir el rango de calidad de agua según el CONDUCTIVITY
# Se crea la columna para los rangos del parámetro CONDUCTIVITY
df02 = (
    df02
    .withColumn(
        "qrCOND",
        F.when(
            (df01.CONDUCTIVITY < 75),  # Agua dulce
            100
        )
        .when(
            (
                (df01.CONDUCTIVITY >= 75) & (df01.CONDUCTIVITY < 150)   # Agua moderada 
            ),
            80
        )
        .when(
            (
                (df01.CONDUCTIVITY >= 150) & (df01.CONDUCTIVITY < 225)  # Agua dura 
            ),
            60
        )
        .when(
            (
                (df01.CONDUCTIVITY >= 225) & (df01.CONDUCTIVITY < 300)  # Agua muy dura 
            ) ,
            40
        )
        .otherwise(0)  # Rango residual
    )
)

In [0]:
#Función para definir el rango de calidad de agua según el BOD
# Se crea la columna para los rangos del parámetro BOD
#Función para definir el rango de calidad de agua según el DO
# Se crea la columna para los rangos del parámetro DO
"""
* Rango de calidad para **Demanda Bioquímica de Oxigeno**. Se crea una nueva columna con los rangos **BOD** de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`BOD = [0.0,3.0)`**
    + 80:  *agua moderada* **`BOD = [3.0, 6.0)`**
    + 60:  *agua dura* **`BOD = [6.0, 80.0)`**
    + 40:  *agua muy dura* **`BOD = [80.0, 125.0)`**
    + 0:   rango agua residuales
"""
df02 = (
    df02
    .withColumn(
        "qrBOD",
        F.when(
            (df01.BOD >= 0) & (df01.BOD <= 3),  # Agua dulce
            100
        )
        .when(
            (
                (df01.BOD >= 3) & (df01.BOD < 6)   # Agua moderada
            ),
            80
        )
        .when(
            (
                (df01.BOD >= 6) & (df01.BOD < 80)  # Agua dura
            ),
            60
        )
        .when(
            (
                (df01.BOD >= 80) & (df01.BOD < 125)  # Agua muy dura
            ),
            40
        )
        .otherwise(0)  # Rango residual
    )
)

In [0]:
#Función para definir el rango de calidad de agua según el NITRATE_N_NITRITE_N
# Se crea la columna para los rangos del parámetro NITRATE_N_NITRITE_N
df02 = (
    df02
    .withColumn(
        "qrNITRATE",
        F.when(
            (df01.NITRATE_N_NITRITE_N < 20),  # Agua dulce
            100
        )
        .when(
            (
                (df01.NITRATE_N_NITRITE_N >= 20) & (df01.NITRATE_N_NITRITE_N < 50)   # Agua moderada 
            ),
            80
        )
        .when(
            (
                (df01.NITRATE_N_NITRITE_N >= 50) & (df01.NITRATE_N_NITRITE_N < 100)  # Agua dura 
            ),
            60
        )
        .when(
            (
                (df01.NITRATE_N_NITRITE_N >= 100) & (df01.NITRATE_N_NITRITE_N < 200)  # Agua muy dura 
            ) ,
            40
        )
        .otherwise(0)  # Rango residual
    )
)

In [0]:
#Función para definir el rango de calidad de agua según el material fecal
# Se crea la columna para los rangos del parámetro FECAL_COLIFORM
"""
* Rango de calidad para **Coliforme Fecal**. Se crea una nueva columna con los rangos FECAL_COLIFORM de la calidad del agua **[2]**. A saber:

    + 100: *agua dulce* **`FECAL_COLIFORM = [0.0,5.0)`**
    + 80:  *agua moderada* **`FECAL_COLIFORM = [5.0, 50.0)`**
    + 60:  *agua dura* **`FECAL_COLIFORM = [50.0, 500.0)`**
    + 40:  *agua muy dura* **`FECAL_COLIFORM = [500.0, 10000.0)`**
    + 0:   rango agua residuales
"""
df02 = (
    df02
    .withColumn(
        "qrFECAL",
        F.when(
            (df01.FECAL_COLIFORM >= 0) & (df01.FECAL_COLIFORM <= 5),  # Agua dulce
            100
        )
        .when(
            (
                (df01.FECAL_COLIFORM >= 5) & (df01.FECAL_COLIFORM < 50)   # Agua moderada
            ),
            80
        )
        .when(
            (
                (df01.FECAL_COLIFORM >= 50) & (df01.FECAL_COLIFORM < 500)  # Agua dura
            ),
            60
        )
        .when(
            (
                (df01.FECAL_COLIFORM >= 500) & (df01.FECAL_COLIFORM < 10000)  # Agua muy dura
            ),
            40
        )
        .otherwise(0)  # Rango residual
    )
)

In [0]:
df02.columns

### **Cálculo de WQI para cada parámetro**
- El peso del agua puede ser calculado por el producto de su volumen por su densidad. La densidad del agua es aproximadamente 1 gramo por centimetro cúbico (g/cm^3)

In [0]:
df03 = df02.withColumn("wpH", F.round(df02.qrPH * 0.165, 4))
df03 = df03.withColumn("wDO", F.round(df02.qrDO * 0.281, 4))
df03 = df03.withColumn("wCOND", F.round(df02.qrCOND * 0.234, 4))
df03 = df03.withColumn("wBOD", F.round(df02.qrBOD * 0.009, 4))
df03 = df03.withColumn("wNITRATE", F.round(df02.qrNITRATE * 0.028, 4))
df03 = df03.withColumn("wFECAL", F.round(df02.qrFECAL * 0.281, 4))

df03.show(5)

%md
### **Cálculo de Índice de Calidad de Agua**

- Se crea la columna WQI <Nota: recordar que son muy pocos los parámetros para un estudio apropiado de Calidad de Agua>

In [0]:
#TODO : sumatoria de los wPARAMETER en df04

%md
* A continuación se clasifica el agua sobre la base de su índice de calidad **[2]**

* Clasificación de calidad del agua:

    + Excelente: *agua dulce* **`WQI = [0.0,25.0]`**
    + Buena:  *agua moderada* **`WQI = (25.0, 50.0]`**
    + Baja:  *agua dura* **`WQI = (50.0, 75.0]`**
    + Muy_Baja:  *agua muy dura* **`WQI = (75.0, 100.0]`**
    + Inadecuada:   rango agua residuales **`WQI > 100.0`**

In [0]:
df05 = df04.withColumn("CALIDAD", F.when((df04.WQI >= 0.0) & (df04.WQI <= 25.0), 'Excelente')
   .when(
       (df04.WQI > 25.0) & (df04.WQI <= 50.0), 'Buena'
       )
   .when(
       (df04.WQI > 50.0) & (df04.WQI <= 75.0), 'Baja'
       )
   .when(
       (df04.WQI > 75.0) & (df04.WQI <= 100.0), 'Muy_Baja'
   )
   .otherwise('Inadecuada'))

df05.show(10)

In [0]:
#TODO: Análisis descriptibvo de calidad del agua por variables

# **Referencias**

* **[1]** Spellman FR. Handbook of Water and Wastewater Treatment Plant Operations. 3rd ed. Boca Raton: CRC Press; 2013.
* **[2]** Summer Kevin. [Water Quality](https://www.intechopen.com/chapters/69568). IntechOpen; DOI 978-1-78985-578-4.2020.
* **[3]** Hammer MJ. Water and Wastewater Technology. 7th ed. Upper Saddle River: Pearson education; 2011.
* **[4]** Tchobanoglous G, Burton FL, Stensel HD. Metcalf & Eddy Wastewater Engineering: Treatment and Reuse. 4th ed. New Delhi: Tata McGraw-Hill Limited; 2003.
* **[5]** Alley ER. Water Quality Control Handbook. Vol. 2. New York: McGraw-Hill; 2007.

* **[6]** Nathanson JA. Basic Environmental Technology: Water Supply. New Delhi: Printice-Hall of India; 2004